# Challenge 8: Gate, report and post-release proof

**EDD: Ship and Feed back. Model upgrade framework, steps 6 and 7.** Everything so far was interactive. Here it becomes a decision made by code, a one-page report anyone can read,
and a check that the release helped. You'll write the gate, run it on the two experiments from challenge 5 (no re-running), print the report, and compare the two production releases.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

from collections import defaultdict

from eval_lab import gate, metrics

## 1. The seven steps, and where you did them

The framework is eval-driven development applied to a model change; the third column is the EDD stage. The same procedure covers a proactive change and a mandated one. Only the trigger (step 1) and the decision (step 6) differ.

In [ ]:
for n, title, where in gate.FRAMEWORK:
    print(f"step {n}: {title:52} {gate.EDD_STAGE[n]:12} {where}")

## 2. The decision rule

The gate compares **quality, cost and latency together, per segment**, and has three outcomes:

- `PASS`: nothing to flag.
- `PASS WITH CONDITIONS`: ship, with named follow-ups (for example monitor one segment, or confirm a cost budget).
- `BLOCKED`: at least one block reason.

The thresholds are **template values** (below); a real team sets them per use case in step 1 of the framework.

In [ ]:
print(json.dumps(gate.THRESHOLDS, indent=2))

Write `decide(base, cand)`, where `base` and `cand` are the summaries from challenge 5 (`overall` and `segments`, each with quality, each score key, cost per example, latency p95, error rate and policy violations).
Return `(outcome, blocks, conditions)`. A reference solution is in the next cell. Change a threshold afterwards and re-run Step 3 to see the outcome move.

In [ ]:
# Reference solution: try writing this yourself first.
def decide(base, cand, t=gate.THRESHOLDS):
    blocks, conditions = [], []
    bo, co = base["overall"], cand["overall"]

    # 1. Each score key: a floor, and a drop against the baseline.
    for key in metrics.KEYS:
        if bo.get(key) is None or co.get(key) is None:
            blocks.append(f"{key}: missing score")
        elif co[key] < t["key_floor"]:
            blocks.append(f"{key}: {co[key]:.2f} is below the floor {t['key_floor']}")
        elif bo[key] - co[key] > t["key_drop_block"]:
            blocks.append(f"{key}: dropped {bo[key]:.2f} -> {co[key]:.2f}")
        elif bo[key] - co[key] > t["key_drop_warn"]:
            conditions.append(f"{key}: dropped {bo[key]:.2f} -> {co[key]:.2f}; review before full rollout")

    # 2. Each segment: an average can hide a regression.
    for seg, b in base["segments"].items():
        c = cand["segments"].get(seg)
        if c is None or b["quality"] is None or c["quality"] is None:
            conditions.append(f"segment {seg}: not measured for the candidate")
            continue
        drop = b["quality"] - c["quality"]
        if drop > t["segment_drop_block"]:
            blocks.append(f"segment {seg}: quality dropped {b['quality']:.2f} -> {c['quality']:.2f}")
        elif drop > t["segment_drop_warn"]:
            conditions.append(f"segment {seg}: quality dropped {b['quality']:.2f} -> {c['quality']:.2f}; monitor this segment")

    # 3. Safety: no new refunds without an eligibility check.
    if co["policy_violations"] > bo["policy_violations"]:
        blocks.append(f"safety: {co['policy_violations']} refunds without an eligibility check (baseline {bo['policy_violations']})")

    # 4. Cost: a big rise needs a quality gain to justify it.
    if bo["cost_per_example"] and co["cost_per_example"] is not None:
        rise = (co["cost_per_example"] / bo["cost_per_example"] - 1) * 100
        gain = (co["quality"] or 0) - (bo["quality"] or 0)
        if rise > t["cost_rise_pct"]:
            if gain < t["min_quality_gain"]:
                blocks.append(f"cost per example up {rise:.0f}% for a quality change of {gain:+.2f}")
            else:
                conditions.append(f"cost per example up {rise:.0f}% for a quality change of {gain:+.2f}; confirm the budget")

    # 5. Latency: a hard budget, and a warning on a big rise.
    p95, base_p95 = co["latency_p95"], bo["latency_p95"]
    if p95 is not None:
        if p95 > t["latency_p95_budget_s"]:
            blocks.append(f"latency p95 {p95:.1f}s is over the {t['latency_p95_budget_s']:.0f}s budget")
        elif base_p95 and p95 > base_p95 * t["latency_rise_warn"]:
            conditions.append(f"latency p95 {base_p95:.1f}s -> {p95:.1f}s")

    outcome = "BLOCKED" if blocks else ("PASS WITH CONDITIONS" if conditions else "PASS")
    return outcome, blocks, conditions

## 3. Run the gate

`gate.run(decide)` finds the newest `baseline-...` and `candidate-...` experiments from challenge 5, reads quality, cost and latency from them, applies your rule, writes `gate_report.json`
and prints the one-page report. (If those experiments are missing it runs them first.) Any outcome is valid: the point is that code made the decision, with reasons.

In [ ]:
assert callable(decide), "decide() must be defined in the blank cell above"
report = gate.run(decide)
assert report["outcome"] in gate.OUTCOMES, "decide() must return an outcome of PASS, PASS WITH CONDITIONS or BLOCKED"
print("\nreused the challenge 5 experiments:", report["reused_experiments"])

## 4. Turn the report into a file

The report above is plain text, so writing it anywhere is one line. In CI you would attach it to the pull request or a ticket; a model-risk review would store it with the sign-offs.
This cell writes it as Markdown, and as JSON for machines:

In [ ]:
from pathlib import Path

md_path = Path(lab_sdk.EXPORT_DIR) / "upgrade_report.md"
md_path.write_text("# Model upgrade report\n\n```\n" + gate.render_report(report) + "\n```\n")
json_path = Path(lab_sdk.EXPORT_DIR) / "gate_report.json"   # already written by the gate
print("wrote", md_path)
print("wrote", json_path, "(", json_path.stat().st_size, "bytes )")

## 5. Automate it

The same gate runs outside a notebook, which is what CI or a scheduler calls. It exits `0` on `PASS` or `PASS WITH CONDITIONS`, and `1` on `BLOCKED`:

```bash
python -m eval_lab.gate && echo "ship it (check the conditions)" || echo "blocked"
```

That version uses the reference rule `eval_lab.gate.default_decide`. Swap in your own by importing `run` and passing your function.
Run it on every prompt or model change, and the dataset you built in challenge 3 becomes the regression suite.

## 6. The same procedure for other migrations

Only the candidate and the trigger change. This track used Claude to Claude; the steps are identical for:

In [ ]:
for item in gate.OTHER_MIGRATIONS:
    print("-", item)

## 7. Prove it after release (step 7)

Offline scores say a change *should* help. Production says whether it did. Every seeded run carries its release in metadata, and feedback attaches to the run, so comparing releases is a group-by
over runs you already have: error rate, process score, and user rating.

In [ ]:
stats = defaultdict(lambda: {"runs": 0, "errors": 0, "auto": [], "rating": []})
roots = lab_sdk.roots(client)
ratings = {f.run_id: f.score for f in client.list_feedback(run_ids=[r.id for r in roots], feedback_key=["user_rating"])}
for r in roots:
    s = stats[lab_sdk.run_meta(r).get("release")]
    s["runs"] += 1
    s["errors"] += int(bool(r.error))
    s["auto"].append(lab_sdk.auto_score(r))
    if r.id in ratings and ratings[r.id] is not None:
        s["rating"].append(ratings[r.id])


def mean(xs):
    return round(sum(xs) / len(xs), 3) if xs else None


releases = {rel: {"runs": s["runs"], "error_rate": round(s["errors"] / s["runs"], 3),
                  "process_score": mean(s["auto"]), "user_rating": mean(s["rating"])}
            for rel, s in sorted(stats.items()) if rel}
lift = round(releases["v1.1"]["process_score"] - releases["v1.0"]["process_score"], 3)
os.makedirs(lab_sdk.EXPORT_DIR, exist_ok=True)
with open(f"{lab_sdk.EXPORT_DIR}/release_report.json", "w") as f:
    json.dump({"releases": releases, "process_score_lift": lift}, f, indent=2)

print(f"{'release':8}{'runs':>6}{'error_rate':>12}{'process_score':>15}{'user_rating':>13}")
for rel, v in releases.items():
    print(f"{rel:8}{v['runs']:>6}{v['error_rate']:>12}{v['process_score']:>15}{str(v['user_rating']):>13}")
print("process score lift, v1.0 -> v1.1:", lift)

The user rating can move the other way: users rate a denied refund low even when denying it was right. That is why a release needs more than one signal, and why challenge 6's disagreement rule exists.
(If you skipped challenge 6, the user rating column is empty.)

## 8. See it in LangSmith

In [ ]:
lab_links.dataset_link(GOLDEN, hint="Experiments tab: the baseline and candidate experiments behind the report")
lab_links.project_link(PROD, hint="group runs by metadata release to see the post-release comparison in the UI")